<a href="https://colab.research.google.com/github/deepak-kumar-da/smart-comment-classifier/blob/main/notebooks/07_transformer_based_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers scikit-learn iterative-stratification joblib tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

import os
import json
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from tqdm.auto import tqdm

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

BASE = "/content/drive/MyDrive/Smart_Comment_Classifier"
PROCESSED = f"{BASE}/datasets/processed"
MODELS = f"{BASE}/models"
REPORTS = f"{BASE}/datasets/reports"

os.makedirs(MODELS, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Transformer training on CPU may be very slow.")

Mounted at /content/drive
Device: cuda
GPU: Tesla T4


In [5]:
#load datasets

spam_df = pd.read_csv(f"{PROCESSED}/uci_spam_final.csv")
tox_df = pd.read_csv(f"{PROCESSED}/jigsaw_toxicity_final.csv")

TOX_LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print("Spam shape:", spam_df.shape)
print("Toxicity shape:", tox_df.shape)

print("\nSpam columns:", spam_df.columns.tolist())
print("Toxicity columns:", tox_df.columns.tolist())

assert "text" in spam_df.columns
assert "label" in spam_df.columns
assert "text" in tox_df.columns

missing_labels = [c for c in TOX_LABELS if c not in tox_df.columns]
assert not missing_labels, f"Missing toxicity columns: {missing_labels}"

spam_df = spam_df.dropna(subset=["text", "label"]).copy()
tox_df = tox_df.dropna(subset=["text"] + TOX_LABELS).copy()

spam_df["text"] = spam_df["text"].astype(str).str.strip()
tox_df["text"] = tox_df["text"].astype(str).str.strip()

spam_df = spam_df[spam_df["text"] != ""].copy()
tox_df = tox_df[tox_df["text"] != ""].copy()

# Convert spam labels into 0/1.
spam_df["label"] = pd.to_numeric(spam_df["label"], errors="coerce")
spam_df = spam_df.dropna(subset=["label"]).copy()
spam_df["label"] = spam_df["label"].astype(int)

assert set(spam_df["label"].unique()).issubset({0, 1}), \
    "Spam labels must be encoded as 0 and 1."

for col in TOX_LABELS:
    tox_df[col] = pd.to_numeric(tox_df[col], errors="coerce")

tox_df = tox_df.dropna(subset=TOX_LABELS).copy()
tox_df[TOX_LABELS] = tox_df[TOX_LABELS].astype(int)

assert all(set(tox_df[c].unique()).issubset({0, 1}) for c in TOX_LABELS)

print("\nFinal spam rows:", len(spam_df))
print("Spam distribution:\n", spam_df["label"].value_counts())

print("\nFinal toxicity rows:", len(tox_df))
print("\nToxicity positive counts:")
print(tox_df[TOX_LABELS].sum())

Spam shape: (1741, 4)
Toxicity shape: (159272, 9)

Spam columns: ['text', 'label', 'source', 'language']
Toxicity columns: ['text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'source', 'language']

Final spam rows: 1741
Spam distribution:
 label
0    904
1    837
Name: count, dtype: int64

Final toxicity rows: 159272

Toxicity positive counts:
toxic            15215
severe_toxic      1585
obscene           8407
threat             469
insult            7835
identity_hate     1394
dtype: int64


In [6]:
# train-test splits

# SPAM: 70% train, 10% validation, 20% test
spam_trainval, spam_test = train_test_split(
    spam_df,
    test_size=0.20,
    random_state=SEED,
    stratify=spam_df["label"]
)

spam_train, spam_val = train_test_split(
    spam_trainval,
    test_size=0.125,
    random_state=SEED,
    stratify=spam_trainval["label"]
)

# TOXICITY: multilabel stratified split
X_tox = tox_df["text"].values
Y_tox = tox_df[TOX_LABELS].values

split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.20, random_state=SEED
)
tox_trainval_idx, tox_test_idx = next(split1.split(X_tox, Y_tox))

X_trainval = X_tox[tox_trainval_idx]
Y_trainval = Y_tox[tox_trainval_idx]

split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.15, random_state=SEED + 1
)
tox_train_idx, tox_val_idx = next(split2.split(X_trainval, Y_trainval))

tox_train = pd.DataFrame({
    "text": X_trainval[tox_train_idx],
    **{
        label: Y_trainval[tox_train_idx, i]
        for i, label in enumerate(TOX_LABELS)
    }
})

tox_val = pd.DataFrame({
    "text": X_trainval[tox_val_idx],
    **{
        label: Y_trainval[tox_val_idx, i]
        for i, label in enumerate(TOX_LABELS)
    }
})

tox_test = pd.DataFrame({
    "text": X_tox[tox_test_idx],
    **{
        label: Y_tox[tox_test_idx, i]
        for i, label in enumerate(TOX_LABELS)
    }
})

print("SPAM")
print("Train:", len(spam_train))
print("Validation:", len(spam_val))
print("Test:", len(spam_test))

print("\nTOXICITY")
print("Train:", len(tox_train))
print("Validation:", len(tox_val))
print("Test:", len(tox_test))

SPAM
Train: 1218
Validation: 174
Test: 349

TOXICITY
Train: 108304
Validation: 19113
Test: 31855


In [7]:
# configure DistilBERT & tokenization

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = np.asarray(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoded.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])
        return item


def make_loader(texts, labels, batch_size, shuffle=False):
    dataset = CommentDataset(
        texts, labels, tokenizer, MAX_LENGTH
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

print("Tokenizer:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer: distilbert-base-uncased
Maximum sequence length: 128


In [8]:
# define training and evaluation

def move_batch(batch):
    return {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }


def save_best_model(model, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


def get_binary_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )),
        "spam_precision": float(precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        )),
        "spam_recall": float(recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        )),
        "spam_f1": float(f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ))
    }


def get_multilabel_metrics(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )),
        "micro_f1": float(f1_score(
            y_true, y_pred, average="micro", zero_division=0
        )),
        "weighted_f1": float(f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ))
    }


def predict_logits(model, loader):
    model.eval()
    all_logits = []

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            batch = move_batch(batch)
            labels = batch.pop("labels")

            outputs = model(**batch)
            all_logits.append(outputs.logits.detach().cpu().numpy())

    return np.vstack(all_logits)

In [9]:
# training DistilBERT for spam detection

spam_output_dir = f"{MODELS}/distilbert_spam"

spam_train_loader = make_loader(
    spam_train["text"],
    spam_train["label"].values,
    batch_size=16,
    shuffle=True
)

spam_val_loader = make_loader(
    spam_val["text"],
    spam_val["label"].values,
    batch_size=16
)

spam_test_loader = make_loader(
    spam_test["text"],
    spam_test["label"].values,
    batch_size=16
)

spam_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(DEVICE)

class_counts = spam_train["label"].value_counts().sort_index()
class_weights = len(spam_train) / (
    2 * torch.tensor(class_counts.values, dtype=torch.float32)
)

criterion_spam = nn.CrossEntropyLoss(
    weight=class_weights.to(DEVICE)
)

optimizer_spam = torch.optim.AdamW(
    spam_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

EPOCHS_SPAM = 3
best_spam_f1 = -1
best_spam_state = None
spam_history = []

for epoch in range(EPOCHS_SPAM):
    spam_model.train()
    total_loss = 0

    for batch in tqdm(spam_train_loader, desc=f"Spam epoch {epoch+1}"):
        batch = move_batch(batch)
        labels = batch.pop("labels").long()

        optimizer_spam.zero_grad()

        logits = spam_model(**batch).logits
        loss = criterion_spam(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(spam_model.parameters(), 1.0)
        optimizer_spam.step()

        total_loss += loss.item()

    val_logits = predict_logits(spam_model, spam_val_loader)
    val_preds = np.argmax(val_logits, axis=1)

    val_metrics = get_binary_metrics(
        spam_val["label"].values, val_preds
    )

    avg_loss = total_loss / max(len(spam_train_loader), 1)

    record = {
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        **val_metrics
    }
    spam_history.append(record)

    print("\n", record)

    if val_metrics["macro_f1"] > best_spam_f1:
        best_spam_f1 = val_metrics["macro_f1"]
        best_spam_state = copy.deepcopy(spam_model.state_dict())

    torch.cuda.empty_cache() if torch.cuda.is_available() else None

spam_model.load_state_dict(best_spam_state)
save_best_model(spam_model, spam_output_dir)

print("\nBest spam validation macro-F1:", best_spam_f1)
print("Saved:", spam_output_dir)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Spam epoch 1:   0%|          | 0/77 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

... (5 outputs left out)

  0%|          | 0/11 [00:00<?, ?it/s]


 {'epoch': 3, 'train_loss': 0.045722202913436504, 'accuracy': 0.9655172413793104, 'macro_f1': 0.9654029692470838, 'spam_precision': 0.9875, 'spam_recall': 0.9404761904761905, 'spam_f1': 0.9634146341463414}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best spam validation macro-F1: 0.9654029692470838
Saved: /content/drive/MyDrive/Smart_Comment_Classifier/models/distilbert_spam


In [10]:
# evaluation

spam_test_logits = predict_logits(spam_model, spam_test_loader)
spam_test_probs = torch.softmax(
    torch.tensor(spam_test_logits), dim=1
).numpy()

spam_test_preds = np.argmax(spam_test_probs, axis=1)
y_spam_test = spam_test["label"].values

spam_test_metrics = get_binary_metrics(
    y_spam_test, spam_test_preds
)

print("DISTILBERT SPAM TEST METRICS")
print(json.dumps(spam_test_metrics, indent=2))

print("\nClassification report:")
print(classification_report(
    y_spam_test,
    spam_test_preds,
    target_names=["Not spam", "Spam"],
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(y_spam_test, spam_test_preds))

  0%|          | 0/22 [00:00<?, ?it/s]

DISTILBERT SPAM TEST METRICS
{
  "accuracy": 0.9742120343839542,
  "macro_f1": 0.9741270664986286,
  "spam_precision": 0.9937888198757764,
  "spam_recall": 0.9523809523809523,
  "spam_f1": 0.9726443768996961
}

Classification report:
              precision    recall  f1-score   support

    Not spam       0.96      0.99      0.98       181
        Spam       0.99      0.95      0.97       168

    accuracy                           0.97       349
   macro avg       0.98      0.97      0.97       349
weighted avg       0.97      0.97      0.97       349


Confusion matrix:
[[180   1]
 [  8 160]]


In [11]:
# toxicity loaders and model preparation

tox_train_loader = make_loader(
    tox_train["text"],
    tox_train[TOX_LABELS].values.astype(np.float32),
    batch_size=8,
    shuffle=True
)

tox_val_loader = make_loader(
    tox_val["text"],
    tox_val[TOX_LABELS].values.astype(np.float32),
    batch_size=8
)

tox_test_loader = make_loader(
    tox_test["text"],
    tox_test[TOX_LABELS].values.astype(np.float32),
    batch_size=8
)

tox_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TOX_LABELS)
).to(DEVICE)

# Class imbalance adjustment: one positive weight per toxicity label.
positive_counts = tox_train[TOX_LABELS].sum(axis=0).values
negative_counts = len(tox_train) - positive_counts

pos_weights = negative_counts / np.maximum(positive_counts, 1)
pos_weights = np.clip(pos_weights, 1.0, 20.0)

pos_weights_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=DEVICE
)

criterion_tox = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights_tensor
)

optimizer_tox = torch.optim.AdamW(
    tox_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print("Positive weights:")
print(dict(zip(TOX_LABELS, pos_weights.round(2))))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Positive weights:
{'toxic': np.float64(9.47), 'severe_toxic': np.float64(20.0), 'obscene': np.float64(17.94), 'threat': np.float64(20.0), 'insult': np.float64(19.33), 'identity_hate': np.float64(20.0)}


In [12]:
# train DistilBERT for multi label toxicity

tox_output_dir = f"{MODELS}/distilbert_toxicity"

EPOCHS_TOX = 2
best_tox_f1 = -1
best_tox_state = None
tox_history = []

y_tox_val = tox_val[TOX_LABELS].values.astype(int)

for epoch in range(EPOCHS_TOX):
    tox_model.train()
    total_loss = 0

    for batch in tqdm(tox_train_loader, desc=f"Toxicity epoch {epoch+1}"):
        batch = move_batch(batch)
        labels = batch.pop("labels").float()

        optimizer_tox.zero_grad()

        logits = tox_model(**batch).logits
        loss = criterion_tox(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(tox_model.parameters(), 1.0)
        optimizer_tox.step()

        total_loss += loss.item()

    val_logits = predict_logits(tox_model, tox_val_loader)
    val_probs = 1 / (1 + np.exp(-np.clip(val_logits, -30, 30)))
    val_preds = (val_probs >= 0.5).astype(int)

    val_metrics = get_multilabel_metrics(
        y_tox_val, val_preds
    )

    avg_loss = total_loss / max(len(tox_train_loader), 1)

    record = {
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        **val_metrics
    }
    tox_history.append(record)

    print("\n", record)

    if val_metrics["macro_f1"] > best_tox_f1:
        best_tox_f1 = val_metrics["macro_f1"]
        best_tox_state = copy.deepcopy(tox_model.state_dict())

    torch.cuda.empty_cache() if torch.cuda.is_available() else None

tox_model.load_state_dict(best_tox_state)
save_best_model(tox_model, tox_output_dir)

print("\nBest toxicity validation macro-F1:", best_tox_f1)
print("Saved:", tox_output_dir)

Toxicity epoch 1:   0%|          | 0/13538 [00:00<?, ?it/s]

  0%|          | 0/2390 [00:00<?, ?it/s]


 {'epoch': 1, 'train_loss': 0.3359907015922728, 'macro_f1': 0.6448867040375372, 'micro_f1': 0.7479975354282193, 'weighted_f1': 0.7580400114409682}


Toxicity epoch 2:   0%|          | 0/13538 [00:00<?, ?it/s]

  0%|          | 0/2390 [00:00<?, ?it/s]


 {'epoch': 2, 'train_loss': 0.2400997027357421, 'macro_f1': 0.6235875807411164, 'micro_f1': 0.7389993972272453, 'weighted_f1': 0.7607496725992177}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best toxicity validation macro-F1: 0.6448867040375372
Saved: /content/drive/MyDrive/Smart_Comment_Classifier/models/distilbert_toxicity


In [13]:
# evaluation

tox_test_logits = predict_logits(tox_model, tox_test_loader)

tox_test_probs = 1 / (
    1 + np.exp(-np.clip(tox_test_logits, -30, 30))
)

y_tox_test = tox_test[TOX_LABELS].values.astype(int)
tox_test_preds_default = (tox_test_probs >= 0.5).astype(int)

default_metrics = get_multilabel_metrics(
    y_tox_test, tox_test_preds_default
)

print("DISTILBERT TOXICITY TEST METRICS — DEFAULT THRESHOLD")
print(json.dumps(default_metrics, indent=2))

print("\nPer-label report:")
for i, label in enumerate(TOX_LABELS):
    print(f"\n{label}")
    print(classification_report(
        y_tox_test[:, i],
        tox_test_preds_default[:, i],
        zero_division=0
    ))

  0%|          | 0/3982 [00:00<?, ?it/s]

DISTILBERT TOXICITY TEST METRICS — DEFAULT THRESHOLD
{
  "macro_f1": 0.6407279707801846,
  "micro_f1": 0.7529455537641077,
  "weighted_f1": 0.7616291532512457
}

Per-label report:

toxic
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     28812
           1       0.75      0.86      0.81      3043

    accuracy                           0.96     31855
   macro avg       0.87      0.92      0.89     31855
weighted avg       0.96      0.96      0.96     31855


severe_toxic
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     31538
           1       0.32      0.83      0.46       317

    accuracy                           0.98     31855
   macro avg       0.66      0.91      0.73     31855
weighted avg       0.99      0.98      0.99     31855


obscene
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     30174
           1       0.70    

In [14]:
# optimize toxicity threshold on val data

tox_val_logits = predict_logits(tox_model, tox_val_loader)

tox_val_probs = 1 / (
    1 + np.exp(-np.clip(tox_val_logits, -30, 30))
)

thresholds = {}

for i, label in enumerate(TOX_LABELS):
    y_true = y_tox_val[:, i]
    scores = tox_val_probs[:, i]

    candidates = np.unique(
        np.concatenate([
            np.linspace(0.10, 0.90, 17),
            np.quantile(scores, np.linspace(0.05, 0.95, 19))
        ])
    )

    best_threshold = 0.5
    best_f1 = -1

    for threshold in candidates:
        preds = (scores >= threshold).astype(int)
        score = f1_score(
            y_true, preds, zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)

    thresholds[label] = best_threshold

print("Validation-optimized thresholds:")
for label, threshold in thresholds.items():
    print(f"{label}: {threshold:.4f}")

  0%|          | 0/2390 [00:00<?, ?it/s]

Validation-optimized thresholds:
toxic: 0.8500
severe_toxic: 0.9000
obscene: 0.9000
threat: 0.6500
insult: 0.8000
identity_hate: 0.6000


In [15]:
# final toxicity test evaluation with optimized threshold

threshold_array = np.array([
    thresholds[label] for label in TOX_LABELS
])

tox_test_preds_tuned = (
    tox_test_probs >= threshold_array
).astype(int)

tuned_metrics = get_multilabel_metrics(
    y_tox_test, tox_test_preds_tuned
)

print("DISTILBERT TOXICITY TEST METRICS — OPTIMIZED THRESHOLDS")
print(json.dumps(tuned_metrics, indent=2))

per_label_results = []

for i, label in enumerate(TOX_LABELS):
    per_label_results.append({
        "label": label,
        "threshold": thresholds[label],
        "precision": precision_score(
            y_tox_test[:, i],
            tox_test_preds_tuned[:, i],
            zero_division=0
        ),
        "recall": recall_score(
            y_tox_test[:, i],
            tox_test_preds_tuned[:, i],
            zero_division=0
        ),
        "f1": f1_score(
            y_tox_test[:, i],
            tox_test_preds_tuned[:, i],
            zero_division=0
        )
    })

tox_per_label_df = pd.DataFrame(per_label_results)
display(tox_per_label_df)

DISTILBERT TOXICITY TEST METRICS — OPTIMIZED THRESHOLDS
{
  "macro_f1": 0.6557193880689568,
  "micro_f1": 0.7763546798029557,
  "weighted_f1": 0.7805381021528867
}


,label,threshold,precision,recall,f1
0,toxic,0.85,0.847895,0.787709,0.816695
1,severe_toxic,0.90,0.419149,0.621451,0.500635
2,obscene,0.90,0.813199,0.864961,0.838282
3,threat,0.65,0.431193,0.500000,0.463054
4,insult,0.80,0.730546,0.802808,0.764974
5,identity_hate,0.60,0.520767,0.584229,0.550676


In [16]:
# comparison -> transformer with classical ml

comparison = pd.DataFrame([
    {
        "task": "Spam",
        "model": "Logistic Regression — TF-IDF",
        "test_f1": 0.95,
        "result_type": "Previously reported"
    },
    {
        "task": "Spam",
        "model": "Linear SVM — TF-IDF",
        "test_f1": 0.94,
        "result_type": "Previously reported"
    },
    {
        "task": "Spam",
        "model": "Naive Bayes — TF-IDF",
        "test_f1": 0.93,
        "result_type": "Previously reported"
    },
    {
        "task": "Spam",
        "model": "DistilBERT",
        "test_f1": spam_test_metrics["spam_f1"],
        "result_type": "Current notebook test result"
    },
    {
        "task": "Toxicity",
        "model": "Linear SVM — TF-IDF",
        "test_f1": 0.6087,
        "result_type": "Previously reported"
    },
    {
        "task": "Toxicity",
        "model": "Logistic Regression — TF-IDF",
        "test_f1": 0.60,
        "result_type": "Previously reported"
    },
    {
        "task": "Toxicity",
        "model": "Naive Bayes — TF-IDF",
        "test_f1": 0.44,
        "result_type": "Previously reported"
    },
    {
        "task": "Toxicity",
        "model": "DistilBERT — default threshold",
        "test_f1": default_metrics["macro_f1"],
        "result_type": "Current notebook test result"
    },
    {
        "task": "Toxicity",
        "model": "DistilBERT — optimized thresholds",
        "test_f1": tuned_metrics["macro_f1"],
        "result_type": "Current notebook test result"
    }
])

display(comparison)

,task,model,test_f1,result_type
0,Spam,Logistic Regression — TF-IDF,0.950000,Previously reported
1,Spam,Linear SVM — TF-IDF,0.940000,Previously reported
2,Spam,Naive Bayes — TF-IDF,0.930000,Previously reported
3,Spam,DistilBERT,0.972644,Current notebook test result
4,Toxicity,Linear SVM — TF-IDF,0.608700,Previously reported
5,Toxicity,Logistic Regression — TF-IDF,0.600000,Previously reported
6,Toxicity,Naive Bayes — TF-IDF,0.440000,Previously reported
7,Toxicity,DistilBERT — default threshold,0.640728,Current notebook test result
8,Toxicity,DistilBERT — optimized thresholds,0.655719,Current notebook test result


In [17]:
# saving models. thresholds and reports

# Save spam metrics and training history
with open(f"{REPORTS}/notebook_07_spam_metrics.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "test_metrics": spam_test_metrics,
        "training_history": spam_history
    }, f, indent=2)

# Save toxicity metrics, thresholds and history
with open(f"{REPORTS}/notebook_07_toxicity_metrics.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "default_threshold_test_metrics": default_metrics,
        "optimized_threshold_test_metrics": tuned_metrics,
        "thresholds": thresholds,
        "training_history": tox_history
    }, f, indent=2)

tox_per_label_df.to_csv(
    f"{REPORTS}/notebook_07_toxicity_per_label.csv",
    index=False
)

comparison.to_csv(
    f"{REPORTS}/notebook_07_model_comparison.csv",
    index=False
)

# Save test predictions for later error analysis
spam_predictions = spam_test[["text", "label"]].copy()
spam_predictions["predicted_label"] = spam_test_preds
spam_predictions["spam_probability"] = spam_test_probs[:, 1]

spam_predictions.to_csv(
    f"{REPORTS}/notebook_07_spam_test_predictions.csv",
    index=False
)

tox_predictions = tox_test[["text"] + TOX_LABELS].copy()

for i, label in enumerate(TOX_LABELS):
    tox_predictions[f"pred_{label}"] = tox_test_preds_tuned[:, i]
    tox_predictions[f"prob_{label}"] = tox_test_probs[:, i]

tox_predictions.to_csv(
    f"{REPORTS}/notebook_07_toxicity_test_predictions.csv",
    index=False
)

print("All Notebook 07 reports saved.")

All Notebook 07 reports saved.


In [18]:
# final summary

required_paths = [
    f"{MODELS}/distilbert_spam",
    f"{MODELS}/distilbert_toxicity",
    f"{REPORTS}/notebook_07_spam_metrics.json",
    f"{REPORTS}/notebook_07_toxicity_metrics.json",
    f"{REPORTS}/notebook_07_toxicity_per_label.csv",
    f"{REPORTS}/notebook_07_model_comparison.csv",
    f"{REPORTS}/notebook_07_spam_test_predictions.csv",
    f"{REPORTS}/notebook_07_toxicity_test_predictions.csv"
]

for path in required_paths:
    print(("OK      " if os.path.exists(path) else "MISSING "), path)

print("\nFinal spam metrics:")
print(spam_test_metrics)

print("\nFinal toxicity metrics:")
print("Default:", default_metrics)
print("Optimized:", tuned_metrics)

print("\nNotebook 07 execution reached the final cell.")

OK       /content/drive/MyDrive/Smart_Comment_Classifier/models/distilbert_spam
OK       /content/drive/MyDrive/Smart_Comment_Classifier/models/distilbert_toxicity
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_spam_metrics.json
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_toxicity_metrics.json
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_toxicity_per_label.csv
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_model_comparison.csv
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_spam_test_predictions.csv
OK       /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_07_toxicity_test_predictions.csv

Final spam metrics:
{'accuracy': 0.9742120343839542, 'macro_f1': 0.9741270664986286, 'spam_precision': 0.9937888198757764, 'spam_recall': 0.9523809523809523, 'spam_f1': 0.97264437